### Import des librairies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency, kruskal, f_oneway, pearsonr, spearmanr
from itertools import combinations
from collections import Counter
import warnings
import os
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# CHARGEMENT DES DONNÉES
# ============================================================================

excel = pd.ExcelFile(r'H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\pneumodetectV3.xlsx')
excel_adresses = pd.ExcelFile(r'H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\NIPs_adresses.xlsx')

# CORRECTION : NE PAS renommer Record ID maintenant
df = pd.read_excel(excel, 'Feuil1')
df_adresses = pd.read_excel(excel_adresses, 'data').rename(columns={'patientNIP': 'NIP'}).drop(columns=['nom_naiss', 'nom', 'prenom'])

# df_geocoded = pd.read_csv(r'H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\patients_geocoded_france')



In [3]:
df_adresses

,NIP,adresse_1,code_postal_1,commune_1
0,9486811,96 RUE DE LA MAREE\n\n,95320.0,SAINT-LEU-LA-FORET
1,1616205,LA BUTTE D AMOUR / BATIMENT D1\n4 PLACE ROSA P...,95470.0,VEMARS
2,1613196,11 RUE CLOS DU CHAPITRE\n\n,60300.0,SENLIS
3,1520329,30 RUE MAXIME COURTIS\n\n,89100.0,SENS
4,787831,12 RUE DE L ARCADE\n\n,94220.0,CHARENTON LE PONT
...,...,...,...,...
3418,8008583,1 ALLEE LOUIS LE NAIN\nLES HOUTRAIS,92500.0,RUEIL-MALMAISON
3419,9406606,1 RUE BEETHOVEN\n\n,75016.0,PARIS
3420,9903009,3 RUE A.FRANCE\n\n,92370.0,CHAVILLE
3421,1816241,70 BOULEVARD DE STRASBOURG\n\n,94130.0,NOGENT-SUR-MARNE


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

def traitement_cohorte_complete(df_initial, df_adresses_initial):
    """
    Pipeline de traitement pour structure avec UNE LIGNE PAR PATIENT
    Avec INNER merge et tracking précis des exclusions
    """
    
    tracking = {
        'etape': [],
        'nb_patients': [],
        'description': []
    }
    
    def add_tracking(etape, df, description):
        tracking['etape'].append(etape)
        tracking['nb_patients'].append(len(df))
        tracking['description'].append(description)
    
    print("="*100)
    print("TRAITEMENT COMPLET DE LA COHORTE")
    print("="*100)
    
    # ========================================================================
    # ÉTAPE 0 : DONNÉES INITIALES
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 0 : DONNÉES INITIALES")
    print("="*100)
    
    df = df_initial.copy()
    df_adresses = df_adresses_initial.copy()
    
    print(f"df_initial : {len(df):,} patients")
    print(f"df_adresses_initial : {len(df_adresses):,} lignes")
    
    add_tracking("0. Initial", df, "Données brutes")
    
    # ========================================================================
    # ÉTAPE 1 : HARMONISATION DES NOMS DE COLONNES
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 1 : HARMONISATION DES COLONNES")
    print("="*100)
    
    # Mapping des colonnes
    colonnes_mapping = {
        'Sexe': 'sexe',
        'Age au diagnostic': 'age_diagnostic',
        'Date du diagnostic de la maladie': 'date_diagnostic',
        'stade de la maladie au diagnostic': 'stade',
        'type histologique': 'type_histologique',
        'Patient fumeur': 'statut_tabagique',
        'Nombre de paquet annee': 'paquet_annee',
        
        # Mutations
        'EGFR Mutation': 'mutation_EGFR',
        'KRAS Mutation': 'mutation_KRAS',
        'BRAF Mutation': 'mutation_BRAF',
        'ROS1 Mutation': 'mutation_ROS1',
        'HER 2 ( ERBB2) Mutation': 'mutation_ERBB2',
        'MET Mutation': 'mutation_MET',
        'ALK Mutation': 'mutation_ALK',
        
        # Réarrangements (traités comme mutations)
        'Rearrangement ALK': 'mutation_ALK_rearr',
        'Rearrangement ROS1': 'mutation_ROS1_rearr',
        'Rearrangement RET': 'mutation_RET',
        'Rearrangement NTRK': 'mutation_NTRK',
        
        # Autres mutations (seront catégorisées comme AUTRES)
        'TP53': 'mutation_TP53',
        'PIK3CA': 'mutation_PIK3CA',
        'NRAS': 'mutation_NRAS'
    }
    
    # Renommer les colonnes présentes
    for old_name, new_name in colonnes_mapping.items():
        if old_name in df.columns:
            df = df.rename(columns={old_name: new_name})
    
    print("✓ Colonnes harmonisées")
    
    # ========================================================================
    # ÉTAPE 2 : FUSION DES RÉARRANGEMENTS AVEC MUTATIONS
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 2 : FUSION RÉARRANGEMENTS = MUTATIONS")
    print("="*100)
    
    # Fusionner ALK
    if 'mutation_ALK' in df.columns and 'mutation_ALK_rearr' in df.columns:
        df['mutation_ALK'] = df['mutation_ALK'].fillna(df['mutation_ALK_rearr'])
        df = df.drop(columns=['mutation_ALK_rearr'])
        print("✓ ALK: Mutation + Réarrangement fusionnés")
    
    # Fusionner ROS1
    if 'mutation_ROS1' in df.columns and 'mutation_ROS1_rearr' in df.columns:
        df['mutation_ROS1'] = df['mutation_ROS1'].fillna(df['mutation_ROS1_rearr'])
        df = df.drop(columns=['mutation_ROS1_rearr'])
        print("✓ ROS1: Mutation + Réarrangement fusionnés")
    
    print(f"\nColonnes mutation finales : {[col for col in df.columns if 'mutation_' in col]}")
    
    # ========================================================================
    # ÉTAPE 3 : NORMALISATION DES VALEURS
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 3 : NORMALISATION DES VALEURS")
    print("="*100)
    
    # Sexe
    if 'sexe' in df.columns:
        df['sexe'] = df['sexe'].astype(str).str.lower().str.strip()
        df['sexe'] = df['sexe'].replace({
            'femme': 'feminin', 'f': 'feminin',
            'homme': 'masculin', 'h': 'masculin', 'm': 'masculin'
        })
        print(f"✓ Sexe: {df['sexe'].value_counts().to_dict()}")
    
    # Date diagnostic
    if 'date_diagnostic' in df.columns:
        df['date_diagnostic'] = pd.to_datetime(df['date_diagnostic'], format='%d/%m/%Y', errors='coerce')
        print(f"✓ Date diagnostic: {df['date_diagnostic'].notna().sum()} valides")
    
    # Âge
    if 'age_diagnostic' in df.columns:
        df['age_diagnostic'] = pd.to_numeric(df['age_diagnostic'], errors='coerce')
        print(f"✓ Âge: {df['age_diagnostic'].notna().sum()} valides")
    
    # Paquet-année
    if 'paquet_annee' in df.columns:
        df['paquet_annee'] = pd.to_numeric(df['paquet_annee'], errors='coerce')
        print(f"✓ Paquet-année: {df['paquet_annee'].notna().sum()} valides")
    
    # Statut tabagique
    if 'statut_tabagique' in df.columns:
        df['statut_tabagique'] = df['statut_tabagique'].astype(str).str.lower().str.strip()
        df.loc[df['statut_tabagique'] == 'ancien fumeur', 'statut_tabagique'] = 'fumeur'
        
        # Si paquet-année > 0 et statut inconnu → fumeur
        condition_fumeur = (df['statut_tabagique'].isin(['non disponible', 'none', 'nan']) & 
                           df['paquet_annee'].notna() & (df['paquet_annee'] > 0))
        df.loc[condition_fumeur, 'statut_tabagique'] = 'fumeur'
        
        # Si paquet-année = 0 et statut inconnu → non fumeur
        condition_non_fumeur = (df['statut_tabagique'].isin(['non disponible', 'none', 'nan']) & 
                               df['paquet_annee'].notna() & (df['paquet_annee'] == 0))
        df.loc[condition_non_fumeur, 'statut_tabagique'] = 'non fumeur'
        
        print(f"✓ Statut tabagique: {df['statut_tabagique'].value_counts().to_dict()}")
    
    # Mutations : normaliser en positive/negative
    colonnes_mutation = [col for col in df.columns if col.startswith('mutation_')]
    for col in colonnes_mutation:
        df[col] = df[col].astype(str).str.lower().str.strip()
        df[col] = df[col].replace({
            'pos': 'positive', 'yes': 'positive', 'oui': 'positive',
            'neg': 'negative', 'no': 'negative', 'non': 'negative', 'wt': 'negative',
            'nan': np.nan, 'none': np.nan
        })
    
    print(f"✓ {len(colonnes_mutation)} colonnes de mutation normalisées")
    
    # ========================================================================
    # ÉTAPE 4 : PSEUDONYMISATION
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 4 : PSEUDONYMISATION")
    print("="*100)
    
    # Vérifier la présence de NIP
    if 'NIP' not in df.columns:
        print("⚠️ Colonne 'NIP' non trouvée - création de pseudos sans NIP")
        df['pseudo_provisoire'] = range(1, len(df) + 1)
        table_correspondance = pd.DataFrame({
            'pseudo_provisoire': df['pseudo_provisoire']
        })
    else:
        # Convertir NIPs
        df['NIP'] = pd.to_numeric(df['NIP'], errors='coerce').astype('Int64')
        df_adresses['NIP'] = pd.to_numeric(df_adresses['NIP'], errors='coerce').astype('Int64')
        
        # Créer table de correspondance
        table_correspondance = df[['NIP']].drop_duplicates().reset_index(drop=True)
        table_correspondance['pseudo_provisoire'] = range(1, len(table_correspondance) + 1)
        
        # Ajouter NIPs des adresses sans correspondance
        nips_adresses_sans_record = df_adresses[~df_adresses['NIP'].isin(table_correspondance['NIP'])][['NIP']].drop_duplicates()
        
        if len(nips_adresses_sans_record) > 0:
            max_pseudo = table_correspondance['pseudo_provisoire'].max()
            nips_adresses_sans_record['pseudo_provisoire'] = range(max_pseudo + 1, max_pseudo + 1 + len(nips_adresses_sans_record))
            table_correspondance = pd.concat([table_correspondance, nips_adresses_sans_record], ignore_index=True)
        
        # Appliquer pseudonymisation
        nip_to_pseudo = dict(zip(table_correspondance['NIP'], table_correspondance['pseudo_provisoire']))
        df['pseudo_provisoire'] = df['NIP'].map(nip_to_pseudo)
        df_adresses['pseudo_provisoire'] = df_adresses['NIP'].map(nip_to_pseudo)
        
        print(f"✓ {len(table_correspondance)} patients pseudonymisés")
        print(f"  - Patients df: {df['pseudo_provisoire'].notna().sum()}")
        print(f"  - Patients df_adresses: {df_adresses['pseudo_provisoire'].notna().sum()}")
    
    add_tracking("4. Après pseudonymisation", df, "Patients avec pseudo")
    
    # ========================================================================
    # ÉTAPE 5 : MERGE AVEC ADRESSES (INNER)
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 5 : MERGE AVEC ADRESSES (INNER)")
    print("="*100)
    
    nb_avant_merge = len(df)
    
    # ✅ INNER MERGE : Garde uniquement les patients avec correspondance dans df_adresses
    df_merged = df.merge(df_adresses, on='pseudo_provisoire', how='inner', suffixes=('', '_adresse'))
    df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]
    
    nb_apres_merge = len(df_merged)
    nb_exclus_sans_correspondance = nb_avant_merge - nb_apres_merge
    
    print(f"✓ Merge INNER effectué:")
    print(f"  - Patients avant merge              : {nb_avant_merge:,}")
    print(f"  - Patients après merge              : {nb_apres_merge:,}")
    print(f"  - Patients exclus (sans correspondance): {nb_exclus_sans_correspondance:,}")
    
    # Vérifier les adresses incomplètes restantes
    vars_adresse = ['adresse_1', 'code_postal_1', 'commune_1']
    if all(v in df_merged.columns for v in vars_adresse):
        nb_adresses_incompletes_post_merge = (~df_merged[vars_adresse].notna().all(axis=1)).sum()
        print(f"  - Adresses incomplètes restantes    : {nb_adresses_incompletes_post_merge:,}")
    
    add_tracking("5. Après merge INNER", df_merged, "Avec correspondance adresse")
    
    # ========================================================================
    # ÉTAPE 6 : EXCLUSION PATIENTS OPPOSÉS
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 6 : EXCLUSION PATIENTS OPPOSÉS")
    print("="*100)
    
    nips_opposes_reinformation = [1090507,2112513,1819574,706700,989299,7804509,8702547,1415948,2019528,1007770,1520752,1615272,1118846,1902577,2009855,8909001]
    nips_opposes_consentement = []
    
    if 'NIP' in df_merged.columns and 'raison_exclusion' in df_initial.columns:
        nips_opposes_consentement = df_initial[
            df_initial['raison_exclusion'].astype(str).str.lower().str.strip() == 'opposé'
        ]['NIP'].dropna().unique().tolist()
    
    nips_opposes = list(set(nips_opposes_reinformation + nips_opposes_consentement))
    
    if len(nips_opposes) > 0 and 'NIP' in df_merged.columns:
        nb_opposes = df_merged['NIP'].isin(nips_opposes).sum()
        df_merged = df_merged[~df_merged['NIP'].isin(nips_opposes)]
        
        # Supprimer également de la table de correspondance
        table_correspondance = table_correspondance[~table_correspondance['NIP'].isin(nips_opposes)]
        
        print(f"✓ {nb_opposes} patients opposés exclus")
    else:
        nb_opposes = 0
        print("✓ Aucun patient opposé à exclure")
    
    add_tracking("6. Après exclusion opposés", df_merged, "Sans opposés")
    
    # ========================================================================
    # ÉTAPE 6B : SUPPRESSION DÉFINITIVE DES NIPs DE LA TABLE PRINCIPALE
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 6B : SUPPRESSION DES NIPs (PSEUDONYMISATION COMPLÈTE)")
    print("="*100)
    
    # Supprimer TOUTES les colonnes contenant NIP de df_merged
    colonnes_nip_a_supprimer = [col for col in df_merged.columns if 'NIP' in col.upper()]
    
    if colonnes_nip_a_supprimer:
        print(f"Colonnes contenant 'NIP' détectées : {colonnes_nip_a_supprimer}")
        df_merged = df_merged.drop(columns=colonnes_nip_a_supprimer)
        print(f"✓ {len(colonnes_nip_a_supprimer)} colonne(s) supprimée(s)")
    else:
        print("✓ Aucune colonne NIP trouvée (déjà supprimée)")
    
    print(f"\n✓ Identifiant unique restant : 'pseudo_provisoire' (entier)")
    print(f"✓ Les NIPs sont conservés UNIQUEMENT dans 'table_correspondance_nip_pseudo.csv'")
    print(f"✓ Pseudonymisation complète : impossible de ré-identifier les patients sans la table")
    
    # ========================================================================
    # ÉTAPE 7 : EXCLUSION SANS DATE DIAGNOSTIC
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 7 : EXCLUSION SANS DATE DIAGNOSTIC")
    print("="*100)
    
    if 'date_diagnostic' in df_merged.columns:
        nb_sans_date = df_merged['date_diagnostic'].isna().sum()
        df_merged = df_merged[df_merged['date_diagnostic'].notna()]
        print(f"✓ {nb_sans_date} patients sans date diagnostic exclus")
    else:
        nb_sans_date = 0
        print("⚠️ Colonne date_diagnostic non trouvée")
    
    add_tracking("7. Après exclusion sans date", df_merged, "Avec date diagnostic")
    
    # ========================================================================
    # ÉTAPE 8 : EXCLUSION ADRESSES INCOMPLÈTES
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 8 : EXCLUSION ADRESSES INCOMPLÈTES")
    print("="*100)
    
    vars_adresse = ['adresse_1', 'code_postal_1', 'commune_1']
    
    if all(v in df_merged.columns for v in vars_adresse):
        # Identifier les adresses incomplètes
        masque_incomplet = ~df_merged[vars_adresse].notna().all(axis=1)
        nb_adresses_incompletes = masque_incomplet.sum()
        
        if nb_adresses_incompletes > 0:
            print(f"\n📋 {nb_adresses_incompletes} patients avec adresses incomplètes détectés")
            
            # Détails par colonne manquante
            print(f"\n📊 Détails des valeurs manquantes :")
            for var in vars_adresse:
                nb_manquant = df_merged[masque_incomplet][var].isna().sum()
                if nb_manquant > 0:
                    print(f"  - {var:20s}: {nb_manquant} valeurs manquantes")
            
            # Sauvegarder les patients exclus
            output_dir = Path('../excels')
            output_dir.mkdir(exist_ok=True)
            
            cols_save = ['pseudo_provisoire', 'sexe', 'age_diagnostic'] + vars_adresse
            cols_save = [c for c in cols_save if c in df_merged.columns]
            
            patients_exclus = df_merged[masque_incomplet][cols_save].copy()
            patients_exclus.to_csv(
                output_dir / 'patients_adresses_incompletes.csv',
                index=False, sep=';', encoding='utf-8-sig'
            )
            print(f"\n💾 Patients exclus sauvegardés dans 'patients_adresses_incompletes.csv'")
        
        # Exclure les patients
        df_merged = df_merged[~masque_incomplet]
        
        print(f"\n✓ {nb_adresses_incompletes} patients avec adresse incomplète exclus")
        print(f"✓ {len(df_merged)} patients restants avec adresse complète")
        
    else:
        nb_adresses_incompletes = 0
        print("⚠️ Colonnes d'adresse non trouvées - étape ignorée")
    
    add_tracking("8. Après exclusion adresses incomplètes", df_merged, "Avec adresse complète")
    
    # ========================================================================
    # ÉTAPE 9 : ENRICHISSEMENT DES DONNÉES
    # ========================================================================
    print("\n" + "="*100)
    print("ÉTAPE 9 : ENRICHISSEMENT DES DONNÉES")
    print("="*100)
    
    df_final = df_merged.copy()
    
    # -------------------------------------------------------------------------
    # 9.1 CATÉGORISATION ÂGE
    # -------------------------------------------------------------------------
    if 'age_diagnostic' in df_final.columns:
        df_final['categorie_age'] = pd.cut(
            df_final['age_diagnostic'], 
            bins=[0, 50, 60, 70, 80, 120],
            labels=['<50', '50-60', '60-70', '70-80', '>80']
        )
        print(f"✓ Catégories d'âge créées:")
        print(df_final['categorie_age'].value_counts().sort_index())
    
    # -------------------------------------------------------------------------
    # 9.2 SIMPLIFICATION DES STADES
    # -------------------------------------------------------------------------
    def simplifier_stade(s):
        if pd.isna(s):
            return 'Non disponible'
        s = str(s).upper().replace('-', '').replace(' ', '')
        
        if  'NON DISPONIBLE' in s:
            return 'Non disponible'
        elif s.startswith('IV'):
            return 'IV'
        elif s.startswith('III'):
            return 'III'
        elif s.startswith('II'):
            return 'II'
        elif s.startswith('I'):
            return 'I'
        else:
            return 'Non disponible'
    
    if 'stade' in df_final.columns:
        df_final['stade'] = df_final['stade'].apply(simplifier_stade)
        print(f"\n✓ Stades simplifiés:")
        print(df_final['stade'].value_counts())
    
    # -------------------------------------------------------------------------
    # 9.3 CATÉGORISATION EXPOSITION TABAGIQUE
    # -------------------------------------------------------------------------

    df_final.loc[df_final['statut_tabagique'] == 'non fumeur', 'paquet_annee'] = 0

    # 2. Création des catégories avec "Non exposé" pour la valeur 0
    # On utilise des bins qui commencent légèrement en dessous de 0 pour isoler le 0 pile
    df_final['exposition_tabagique'] = pd.cut(
        df_final['paquet_annee'],
        bins=[-0.1, 0, 10, 20, 40, float('inf')],
        labels=['Non exposé', 'Faible (1-10)', 'Modérée (10-20)', 'Forte (20-40)', 'Très forte (>40)'],
        include_lowest=True
)

    print(f"\n✓ Exposition tabagique catégorisée (incluant non-exposés) :")
    print(df_final['exposition_tabagique'].value_counts().sort_index())
    
    # -------------------------------------------------------------------------
    # 9.4 PROFIL COMPLET DES MUTATIONS (MULTI-MUTATIONS)
    # -------------------------------------------------------------------------
    print("\n" + "-"*80)
    print("PROFIL COMPLET DES MUTATIONS (MULTI-MUTATIONS)")
    print("-"*80)

    # Votre liste précise desmutations associées aux fumeurs
    MUTATIONS_PRINCIPALES =  ['EGFR', 'ALK', 'ROS1', 'ERBB2', 'RET', 'MET', 'NTRK']

    # Identifier toutes les colonnes de mutations présentes (ex: mutation_EGFR, mutation_TP53...)
    colonnes_mutation = [col for col in df_final.columns if col.startswith('mutation_')]

    # Initialisation des colonnes de synthèse
    df_final['mutation_AUTRES'] = 'negative'
    df_final['mutation'] = 'NO MUTATION'
    df_final['a_mutation_driver'] = False  # Création de la colonne qui causait l'erreur

    for idx, row in df_final.iterrows():
        mutations_positives_du_patient = []
        has_driver = False
        has_autres = False
        
        for col in colonnes_mutation:
            # Extraire le nom du gène (ex: mutation_EGFR -> EGFR)
            nom_gene = col.replace('mutation_', '').upper().strip()
            valeur = str(row[col]).lower().strip()
            
            # Si le patient est positif pour cette mutation
            if valeur in ['positive', 'pos', 'oui', 'yes']:
                mutations_positives_du_patient.append(nom_gene)
                
                # Catégorisation selon votre liste
                if nom_gene in MUTATIONS_PRINCIPALES:
                    has_driver = True
                else:
                    has_autres = True
        
        # Mise à jour des indicateurs pour ce patient
        df_final.at[idx, 'a_mutation_driver'] = has_driver
        
        if has_autres:
            df_final.at[idx, 'mutation_AUTRES'] = 'positive'

        
        # Construction de la chaîne "Profil" (ex: "EGFR,KRAS" ou "ALK" ou "NO MUTATION")
        if mutations_positives_du_patient:
            df_final.at[idx, 'mutation'] = ",".join(sorted(set(mutations_positives_du_patient)))
        else:
            df_final.at[idx, 'mutation'] = 'NO MUTATION'

    # --- AFFICHAGE DES RÉSULTATS ---
    print("\n📊 Distribution des profils mutationnels détectés :")
    dist = df_final['mutation'].value_counts()
    for mut, count in dist.items():
        pct = count / len(df_final) * 100
        print(f"  {mut:30s}: {count:4d} ({pct:5.1f}%)")

    print(f"\n✅ Patients avec au moins une mutation driver : {df_final['a_mutation_driver'].sum()} "
        f"({df_final['a_mutation_driver'].mean()*100:.1f}%)")
    
    add_tracking("9. Après enrichissement", df_final, "Cohorte finale enrichie")
    
    # ========================================================================
    # STATISTIQUES FINALES
    # ========================================================================
    print("\n" + "="*100)
    print("STATISTIQUES FINALES")
    print("="*100)
    
    df_tracking = pd.DataFrame(tracking)
    print("\n" + df_tracking.to_string(index=False))
    
    print("\n" + "="*100)
    print("COMPLÉTUDE DES VARIABLES")
    print("="*100)
    
    vars_a_verifier = ['sexe', 'age_diagnostic', 'date_diagnostic', 'statut_tabagique', 
                       'paquet_annee', 'stade', 'type_histologique']
    
    for var in vars_a_verifier:
        if var in df_final.columns:
            nb_valides = df_final[var].notna().sum()
            pct = nb_valides / len(df_final) * 100
            print(f"  {var:20s}: {nb_valides:4d}/{len(df_final):4d} ({pct:5.1f}%)")
    
    # ========================================================================
    # RÉSUMÉ FINAL (CORRIGÉ)
    # ========================================================================
    nb_initial = len(df_initial)
    nb_final = len(df_final)
    
    print("\n" + "="*100)
    print("RÉSUMÉ FINAL")
    print("="*100)
    print(f"""
DONNÉES INITIALES : {nb_initial:,} patients

EXCLUSIONS (séquentielles) :
  1. Sans correspondance dans df_adresses  : {nb_exclus_sans_correspondance:,}
  2. Patients opposés                      : {nb_opposes:,}
  3. Sans date diagnostic                  : {nb_sans_date:,}
  4. Avec adresse incomplète               : {nb_adresses_incompletes:,}
  ───────────────────────────────────────────────────────────
  TOTAL EXCLUS                             : {nb_initial - nb_final:,} ({(nb_initial - nb_final)/nb_initial*100:.1f}%)

COHORTE FINALE : {nb_final:,} patients ({nb_final/nb_initial*100:.1f}%)
    """)
    
    # Vérification arithmétique
    total_calcule = nb_exclus_sans_correspondance + nb_opposes + nb_sans_date + nb_adresses_incompletes
    total_reel = nb_initial - nb_final
    
    if total_calcule == total_reel:
        print(f"✅ Vérification arithmétique : OK ({total_calcule} = {total_reel})")
    else:
        print(f"⚠️  Vérification arithmétique : ATTENTION")
        print(f"   Somme calculée : {total_calcule}")
        print(f"   Différence réelle : {total_reel}")
        print(f"   Écart : {abs(total_calcule - total_reel)}")
    
    # ========================================================================
    # SAUVEGARDES
    # ========================================================================
    print("\n" + "="*100)
    print("SAUVEGARDE DES FICHIERS")
    print("="*100)
    
    # Créer dossier si nécessaire
    output_dir = Path('../excels')
    output_dir.mkdir(exist_ok=True)
    
    try:
        # Tracking
        df_tracking.to_csv(output_dir / 'tracking_parcours_patients.csv', 
                          index=False, sep=';', encoding='utf-8-sig')
        
        # Table correspondance
        if 'NIP' in table_correspondance.columns:
            table_correspondance.to_csv(output_dir / 'table_correspondance_nip_pseudo.csv', 
                                       index=False, sep=';', encoding='utf-8-sig')
        
        # Données finales
        df_final_save = df_final.copy()
        
        # ✅ VÉRIFICATION FINALE : Aucun NIP dans la table finale
        colonnes_avec_nip = [col for col in df_final_save.columns if 'NIP' in col.upper()]
        if colonnes_avec_nip:
            print(f"\n⚠️  ATTENTION : Colonnes NIP détectées dans df_final : {colonnes_avec_nip}")
            print(f"⚠️  Suppression de ces colonnes pour garantir la pseudonymisation...")
            df_final_save = df_final_save.drop(columns=colonnes_avec_nip)
            print(f"✓ Colonnes NIP supprimées de la sauvegarde finale")
        
        print(f"\n✅ VÉRIFICATION : Aucune colonne NIP dans la cohorte finale")
        print(f"✅ Seul identifiant : pseudo_provisoire")
        
        # Convertir dates en string pour CSV
        for col in df_final_save.columns:
            if pd.api.types.is_datetime64_any_dtype(df_final_save[col]):
                df_final_save[col] = df_final_save[col].dt.strftime('%Y-%m-%d')
        
        df_final_save.to_csv(output_dir / 'cohorte_finale_avec_adresses.csv', 
                            index=False, sep=';', encoding='utf-8-sig')
        
        print("\n✓ Fichiers sauvegardés:")
        print(f"  - {output_dir / 'tracking_parcours_patients.csv'}")
        print(f"  - {output_dir / 'cohorte_finale_avec_adresses.csv'}")
        if 'NIP' in table_correspondance.columns:
            print(f"  - {output_dir / 'table_correspondance_nip_pseudo.csv'}")
        
    except Exception as e:
        print(f"✗ Erreur lors de la sauvegarde: {e}")
    
    return df_final, df_tracking, table_correspondance


# ============================================================================
# EXEMPLE D'UTILISATION
# ============================================================================

if __name__ == "__main__":
    

    
    # Exécuter le traitement
    df_final, tracking, table_corresp = traitement_cohorte_complete(df, df_adresses)
    



TRAITEMENT COMPLET DE LA COHORTE

ÉTAPE 0 : DONNÉES INITIALES
df_initial : 3,424 patients
df_adresses_initial : 3,423 lignes

ÉTAPE 1 : HARMONISATION DES COLONNES
✓ Colonnes harmonisées

ÉTAPE 2 : FUSION RÉARRANGEMENTS = MUTATIONS
✓ ALK: Mutation + Réarrangement fusionnés
✓ ROS1: Mutation + Réarrangement fusionnés

Colonnes mutation finales : ['mutation_EGFR', 'mutation_KRAS', 'mutation_BRAF', 'mutation_ROS1', 'mutation_ERBB2', 'mutation_MET', 'mutation_ALK', 'mutation_RET', 'mutation_NTRK', 'mutation_TP53', 'mutation_PIK3CA', 'mutation_NRAS']

ÉTAPE 3 : NORMALISATION DES VALEURS
✓ Sexe: {'masculin': 1876, 'feminin': 1547, 'nan': 1}
✓ Date diagnostic: 3424 valides
✓ Âge: 3424 valides
✓ Paquet-année: 2676 valides
✓ Statut tabagique: {'fumeur': 2857, 'non fumeur': 454, 'non disponible': 113}
✓ 12 colonnes de mutation normalisées

ÉTAPE 4 : PSEUDONYMISATION
✓ 3433 patients pseudonymisés
  - Patients df: 3424
  - Patients df_adresses: 3423

ÉTAPE 5 : MERGE AVEC ADRESSES (INNER)
✓ Merge INN

In [5]:
table_corresp

,NIP,pseudo_provisoire
0,9486811,1
1,1616205,2
2,1613196,3
3,1520329,4
4,787831,5
...,...,...
3428,2010937,3429
3429,1907682,3430
3430,1416502,3431
3431,1719481,3432


In [6]:
df_final

,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,mutation_NRAS,pseudo_provisoire,adresse_1,code_postal_1,commune_1,categorie_age,exposition_tabagique,mutation_AUTRES,mutation,a_mutation_driver
0,feminin,79,2016-09-12,Non disponible,Carcinome epidermoide,fumeur,50.0,NaN,NaN,NaN,...,NaN,1,96 RUE DE LA MAREE\n\n,95320.0,SAINT-LEU-LA-FORET,70-80,Très forte (>40),negative,NO MUTATION,False
1,masculin,47,2016-10-04,Non disponible,Carcinome epidermoide,fumeur,30.0,NaN,NaN,positive,...,NaN,2,LA BUTTE D AMOUR / BATIMENT D1\n4 PLACE ROSA P...,95470.0,VEMARS,<50,Forte (20-40),positive,BRAF,False
2,masculin,67,2016-07-21,Non disponible,Carcinome epidermoide,fumeur,40.0,NaN,NaN,NaN,...,NaN,3,11 RUE CLOS DU CHAPITRE\n\n,60300.0,SENLIS,60-70,Forte (20-40),negative,NO MUTATION,False
3,feminin,57,2015-10-22,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,NaN,4,30 RUE MAXIME COURTIS\n\n,89100.0,SENS,50-60,Forte (20-40),positive,KRAS,False
4,feminin,62,2007-11-15,Non disponible,Carcinome epidermoide,fumeur,44.0,NaN,NaN,NaN,...,NaN,5,12 RUE DE L ARCADE\n\n,94220.0,CHARENTON LE PONT,60-70,Très forte (>40),negative,NO MUTATION,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3409,feminin,85,2023-02-17,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,NaN,3420,1 ALLEE LOUIS LE NAIN\nLES HOUTRAIS,92500.0,RUEIL-MALMAISON,>80,Forte (20-40),positive,KRAS,False
3410,masculin,71,2023-04-05,Non disponible,Adenocarcinome,fumeur,40.0,NaN,NaN,NaN,...,NaN,3421,1 RUE BEETHOVEN\n\n,75016.0,PARIS,70-80,Forte (20-40),negative,MET,True
3411,feminin,64,2023-03-07,Non disponible,Carcinome indifferencie,fumeur,50.0,NaN,positive,NaN,...,NaN,3422,3 RUE A.FRANCE\n\n,92370.0,CHAVILLE,60-70,Très forte (>40),positive,KRAS,False
3412,feminin,67,2018-09-21,Non disponible,Adenocarcinome,fumeur,60.0,NaN,positive,NaN,...,NaN,3423,70 BOULEVARD DE STRASBOURG\n\n,94130.0,NOGENT-SUR-MARNE,60-70,Très forte (>40),positive,"KRAS,TP53",False


In [7]:
df_adresses_clean = pd.read_csv(r'H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\excels\cohorte_finale_avec_adresses.csv', sep=';')

In [8]:
df_adresses_clean

,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,mutation_NRAS,pseudo_provisoire,adresse_1,code_postal_1,commune_1,categorie_age,exposition_tabagique,mutation_AUTRES,mutation,a_mutation_driver
0,feminin,79,2016-09-12,Non disponible,Carcinome epidermoide,fumeur,50.0,NaN,NaN,NaN,...,NaN,1,96 RUE DE LA MAREE\n\n,95320.0,SAINT-LEU-LA-FORET,70-80,Très forte (>40),negative,NO MUTATION,False
1,masculin,47,2016-10-04,Non disponible,Carcinome epidermoide,fumeur,30.0,NaN,NaN,positive,...,NaN,2,LA BUTTE D AMOUR / BATIMENT D1\n4 PLACE ROSA P...,95470.0,VEMARS,<50,Forte (20-40),positive,BRAF,False
2,masculin,67,2016-07-21,Non disponible,Carcinome epidermoide,fumeur,40.0,NaN,NaN,NaN,...,NaN,3,11 RUE CLOS DU CHAPITRE\n\n,60300.0,SENLIS,60-70,Forte (20-40),negative,NO MUTATION,False
3,feminin,57,2015-10-22,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,NaN,4,30 RUE MAXIME COURTIS\n\n,89100.0,SENS,50-60,Forte (20-40),positive,KRAS,False
4,feminin,62,2007-11-15,Non disponible,Carcinome epidermoide,fumeur,44.0,NaN,NaN,NaN,...,NaN,5,12 RUE DE L ARCADE\n\n,94220.0,CHARENTON LE PONT,60-70,Très forte (>40),negative,NO MUTATION,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3399,feminin,85,2023-02-17,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,NaN,3420,1 ALLEE LOUIS LE NAIN\nLES HOUTRAIS,92500.0,RUEIL-MALMAISON,>80,Forte (20-40),positive,KRAS,False
3400,masculin,71,2023-04-05,Non disponible,Adenocarcinome,fumeur,40.0,NaN,NaN,NaN,...,NaN,3421,1 RUE BEETHOVEN\n\n,75016.0,PARIS,70-80,Forte (20-40),negative,MET,True
3401,feminin,64,2023-03-07,Non disponible,Carcinome indifferencie,fumeur,50.0,NaN,positive,NaN,...,NaN,3422,3 RUE A.FRANCE\n\n,92370.0,CHAVILLE,60-70,Très forte (>40),positive,KRAS,False
3402,feminin,67,2018-09-21,Non disponible,Adenocarcinome,fumeur,60.0,NaN,positive,NaN,...,NaN,3423,70 BOULEVARD DE STRASBOURG\n\n,94130.0,NOGENT-SUR-MARNE,60-70,Très forte (>40),positive,"KRAS,TP53",False


In [9]:
[1090507,2112513,1819574,706700,989299,7804509,8702547,1415948,2019528,1007770,1520752,1615272,1118846,1902577,2009855,8909001]

[1090507,
 2112513,
 1819574,
 706700,
 989299,
 7804509,
 8702547,
 1415948,
 2019528,
 1007770,
 1520752,
 1615272,
 1118846,
 1902577,
 2009855,
 8909001]